<a href="https://colab.research.google.com/github/epi24/multimodal-meme-analysis/blob/main/multimodal_image_and_ocr_NEW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import CLIPModel, CLIPProcessor
from torchvision import transforms
from tqdm.auto import tqdm
from PIL import Image
import gc

PATH_TRAIN_JSON = '/content/drive/MyDrive/meme_train.json'
PATH_VAL_JSON   = '/content/drive/MyDrive/meme_val.json'
PATH_IMAGES     = '/content/drive/MyDrive/all_memes/kym_memes'
SAVE_DIR        = '/content/drive/MyDrive/multimodal_image_and_ocr_NEW'
MODEL_NAME      = 'multimodal_model_image_and_ocr_text_clip'


CHECKPOINT_PATH = '/content/drive/MyDrive/multimodal_model_image_and_ocr_text_clip_2/multimodal_model_image_and_ocr_text_clip_epoch_0.pth'
# HYPERPARAMETER
EPOCHS        = 20
LEARNING_RATE = 1e-5
BATCH_SIZE = 2500
NUM_WORKERS = 12
PREFETCH_FACTOR = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.environ["TOKENIZERS_PARALLELISM"] = "false"


class MultimodalDataset(Dataset):
    def __init__(self, json_path, img_base_path, processor, is_train=True):
        self.processor = processor
        self.img_base_path = img_base_path
        self.samples = []
        self.label_map = {}
        self.id_to_label = {}

        # Augmentation nur fürs Training
        if is_train:
            self.transform = transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(degrees=10),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
            ])
        else:
            self.transform = None

        print(f"--- [DATASET] Lade {json_path.split('/')[-1]}... ---")
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        # Labels ermitteln (alphabetisch sortieren!)
        unique_labels = sorted(list(set(item['label'] for item in raw_data)))
        for idx, label in enumerate(unique_labels):
            self.label_map[label] = idx
            self.id_to_label[idx] = label

        # Map speichern (Wichtig für spätere Evaluation)
        with open(os.path.join(SAVE_DIR, f"{MODEL_NAME}_map.json"), 'w') as f:
            json.dump(self.id_to_label, f)

        for item in tqdm(raw_data, desc="Lade Metadaten"):
            label_str = item.get('label')
            filename = item.get('filename')

            # --- TEXT HOLEN ---
            ocr_text = item.get('text', "").strip()
            # RAM-Trick: Sofort kürzen. CLIP verarbeitet eh nur 77 Token (ca. 300 Zeichen)
            if len(ocr_text) > 300: ocr_text = ocr_text[:300]
            if len(ocr_text) < 2: ocr_text = "meme text"

            full_img_path = os.path.join(self.img_base_path, label_str, filename)

            # Nur hinzufügen, wenn Bild existiert
            if not os.path.exists(full_img_path): continue

            self.samples.append({
                'img_path': full_img_path,
                'ocr_text': ocr_text,
                'label': self.label_map[label_str]
            })

        print(f"-> Bereit: {len(self.samples)} Multimodale Paare geladen.\n")

        # RAM aufräumen
        del raw_data
        gc.collect()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # 1. Bild verarbeiten
        try:
            image = Image.open(sample['img_path']).convert("RGB")
            if self.transform: image = self.transform(image)
        except:
            return self.__getitem__((idx + 1) % len(self.samples))

        # 2. Processor für Bild UND Text
        img_inputs = self.processor(images=image, return_tensors="pt")
        text_inputs = self.processor(
            text=[sample['ocr_text']],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=77
        )

        return {
            'pixel_values': img_inputs['pixel_values'].squeeze(0),
            'input_ids': text_inputs['input_ids'].squeeze(0),
            'attention_mask': text_inputs['attention_mask'].squeeze(0),
            'label': torch.tensor(sample['label'], dtype=torch.long)
        }

# ==========================================
# 3. ARCHITEKTUR (MULTIMODAL FUSION)
# ==========================================
class MultimodalFusionNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

        # Backbone einfrieren
        for param in self.clip.parameters(): param.requires_grad = False

        # Letzte Schichten BEIDER Encoder auftauen (Partial Unfreezing)
        for param in self.clip.vision_model.encoder.layers[-1].parameters(): param.requires_grad = True
        for param in self.clip.text_model.encoder.layers[-1].parameters(): param.requires_grad = True
        for name, param in self.clip.named_parameters():
            if "layer_norm" in name: param.requires_grad = True

        # Classifier Head: 512 (Bild) + 512 (Text) = 1024
        self.fusion_head = nn.Sequential(
            nn.Linear(1024, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        # 1. Bild-Features
        vision_out = self.clip.vision_model(pixel_values=pixel_values)
        img_embeds = self.clip.visual_projection(vision_out[1])

        # 2. Text-Features
        text_out = self.clip.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_embeds = self.clip.text_projection(text_out[1])

        # 3. Fusion (Zusammenkleben)
        combined = torch.cat((img_embeds, text_embeds), dim=1)

        # 4. Klassifizieren
        return self.fusion_head(combined)

def run_multimodal_training():
    print("--- Start Multimodal Training (Standard LR) ---")

    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    train_dataset = MultimodalDataset(PATH_TRAIN_JSON, PATH_IMAGES, processor, is_train=True)
    val_dataset   = MultimodalDataset(PATH_VAL_JSON, PATH_IMAGES, processor, is_train=False)

    num_classes = len(train_dataset.label_map)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=PREFETCH_FACTOR)
    val_loader   = DataLoader(val_dataset,batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=PREFETCH_FACTOR)

    model = MultimodalFusionNet(num_classes).to(DEVICE)

    # --- STANDARD OPTIMIZER (Alle Parameter lernen mit derselben Rate) ---
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda')

    start_epoch = 0

    if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
        print(f"\n[INFO] Lade Checkpoint: {CHECKPOINT_PATH}")
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

        if "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            start_epoch = checkpoint["epoch"]
            print(f"[SUCCESS] Geladen! Starte ab Epoche {start_epoch+1}.")
        else:
            model.load_state_dict(checkpoint)
            print("[WARNUNG] Alter Checkpoint (Nur Gewichte).")

    print(f"\nStarte Training bis Epoche {EPOCHS}...\n")

    for epoch in range(start_epoch, EPOCHS):
        print(epoch)
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoche {epoch+1}/{EPOCHS} [Train]")

        for batch in pbar:
            optimizer.zero_grad()

            pixel_values = batch['pixel_values'].to(DEVICE)
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            with torch.amp.autocast('cuda'):
                logits = model(pixel_values, input_ids, attention_mask)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        del pixel_values, input_ids, attention_mask, labels, logits
        gc.collect()
        torch.cuda.empty_cache()

        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoche {epoch+1} [Valid]", leave=False):
                pixel_values = batch['pixel_values'].to(DEVICE)
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['label'].to(DEVICE)

                with torch.amp.autocast('cuda'):
                    logits = model(pixel_values, input_ids, attention_mask)

                _, preds = torch.max(logits, 1)
                val_total += labels.size(0)
                val_correct += (preds == labels).sum().item()

                del pixel_values, input_ids, attention_mask, labels, logits

        val_acc = val_correct / val_total
        print(f" -> Resultat E{epoch+1}: Val Acc: {val_acc:.2%}")

        checkpoint_dict = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }

        torch.save(checkpoint_dict, os.path.join(SAVE_DIR, f"{MODEL_NAME}_epoch_{epoch+1}.pth"))

if __name__ == "__main__":
    run_multimodal_training()

--- Start Multimodal Training (Standard LR) ---
--- [DATASET] Lade meme_train.json... ---


Lade Metadaten:   0%|          | 0/75765 [00:00<?, ?it/s]

-> Bereit: 75763 Multimodale Paare geladen.

--- [DATASET] Lade meme_val.json... ---


Lade Metadaten:   0%|          | 0/9402 [00:00<?, ?it/s]

-> Bereit: 9402 Multimodale Paare geladen.



...

Epoche 19 [Valid]:   0%|          | 0/4 [00:00<?, ?it/s]

 -> Resultat E19: Val Acc: 55.13%
19


Epoche 20/20 [Train]:   0%|          | 0/31 [00:00<?, ?it/s]

Epoche 20 [Valid]:   0%|          | 0/4 [00:00<?, ?it/s]

 -> Resultat E20: Val Acc: 56.40%


In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor
from tqdm.auto import tqdm
from PIL import Image
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd


# --- ANGEPASSTE PFADE AUS ZELLE 1 ---
SAVE_DIR        = '/content/drive/MyDrive/multimodal_image_and_ocr_NEW'
MODEL_NAME      = 'multimodal_model_image_and_ocr_text_clip'

CHECKPOINT_PATH = os.path.join(SAVE_DIR, f'{MODEL_NAME}_epoch_20.pth')
PATH_TEST_JSON = '/content/drive/MyDrive/meme_test.json'
PATH_LABEL_MAP  = os.path.join(SAVE_DIR, f'{MODEL_NAME}_map.json')
PATH_IMAGES     = '/content/drive/MyDrive/all_memes/kym_memes'
OUTPUT_CSV      = 'multimodal_image_and_ocr_text_evaluation_results.csv'

BATCH_SIZE = 2500
NUM_WORKERS = 12
PREFETCH_FACTOR = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ==========================================
# 2. DATASET (BILD + TEXT AUF TEST-DATEN)
# ==========================================
class EvalMultimodalDataset(Dataset):
    def __init__(self, json_path, img_base_path, label_map_path, processor):
        self.processor = processor
        self.img_base_path = img_base_path
        self.samples = []

        # --- LADE DIE OFFIZIELLE LABEL MAP ---
        print(f"Lade offizielle Label-Map: {label_map_path.split('/')[-1]}")
        with open(label_map_path, 'r', encoding='utf-8') as f:
            loaded_map = json.load(f)
            self.id_to_label = {int(k): v for k, v in loaded_map.items()}
            self.label_map = {v: int(k) for k, v in loaded_map.items()}

        print(f"--- [EVAL DATASET] Lade Test-Daten aus {json_path.split('/')[-1]}... ---")
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        for item in tqdm(raw_data, desc="Lade Daten"):
            label_str = item.get('label')

            # Unbekannte Klassen im Testset ignorieren
            if label_str not in self.label_map:
                continue

            filename = item.get('filename')
            full_img_path = os.path.join(self.img_base_path, label_str, filename)

            if not os.path.exists(full_img_path): continue

            # --- TEXT HOLEN UND BEREINIGEN ---
            ocr_text = item.get('text', "").strip()
            if len(ocr_text) > 300: ocr_text = ocr_text[:300]
            if len(ocr_text) < 2: ocr_text = "meme text"

            self.samples.append({
                'img_path': full_img_path,
                'ocr_text': ocr_text,
                'label': self.label_map[label_str],
                'filename': filename
            })

        print(f"-> Bereit: {len(self.samples)} multimodale Test-Paare geladen.\n")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        try:
            image = Image.open(sample['img_path']).convert("RGB")
        except:
            return self.__getitem__((idx + 1) % len(self.samples))

        # Processor für Bild UND Text aufrufen
        img_inputs = self.processor(images=image, return_tensors="pt")
        text_inputs = self.processor(
            text=[sample['ocr_text']],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=77
        )

        return {
            'pixel_values': img_inputs['pixel_values'].squeeze(0),
            'input_ids': text_inputs['input_ids'].squeeze(0),
            'attention_mask': text_inputs['attention_mask'].squeeze(0),
            'label': torch.tensor(sample['label'], dtype=torch.long),
            'filename': sample['filename'],
            'raw_text': sample['ocr_text']
        }

# ==========================================
# 3. ARCHITEKTUR (MUSS IDENTISCH ZUM TRAINING SEIN)
# ==========================================
class MultimodalFusionNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

        self.fusion_head = nn.Sequential(
            nn.Linear(1024, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        vision_out = self.clip.vision_model(pixel_values=pixel_values)
        img_embeds = self.clip.visual_projection(vision_out[1])

        text_out = self.clip.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_embeds = self.clip.text_projection(text_out[1])

        # Late Fusion (Konkatenation)
        combined = torch.cat((img_embeds, text_embeds), dim=1)
        return self.fusion_head(combined)

# ==========================================
# 4. EVALUATION LOOP
# ==========================================
def run_multimodal_evaluation():
    print("--- Starte Multimodale Test-Evaluation ---")

    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    eval_dataset = EvalMultimodalDataset(PATH_TEST_JSON, PATH_IMAGES, PATH_LABEL_MAP, processor)
    eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, prefetch_factor=PREFETCH_FACTOR)

    num_classes = len(eval_dataset.label_map)
    model = MultimodalFusionNet(num_classes).to(DEVICE)

    # --- GEWICHTE LADEN ---
    if os.path.exists(CHECKPOINT_PATH):
        print(f"Lade Checkpoint: {CHECKPOINT_PATH}")
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

        if "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
        else:
            model.load_state_dict(checkpoint)
        print("[SUCCESS] Gewichte erfolgreich geladen.")
    else:
        print(f"[ERROR] Checkpoint nicht gefunden: {CHECKPOINT_PATH}")
        return

    model.eval()

    results_list = []
    all_preds = []
    all_labels = []

    print("Berechne Vorhersagen auf ungesehenen Daten...")

    with torch.no_grad():
        for batch in tqdm(eval_loader):
            pixel_values = batch['pixel_values'].to(DEVICE)
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            filenames = batch['filename']
            raw_texts = batch['raw_text']

            with torch.amp.autocast('cuda'):
                logits = model(pixel_values, input_ids, attention_mask)

            probs = torch.softmax(logits, dim=1)
            confidences, preds = torch.max(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            for i in range(len(filenames)):
                pred_idx = preds[i].item()
                true_idx = labels[i].item()

                # Bereinige den Text für die CSV
                clean_text = raw_texts[i].replace("\n", " ").replace(";", ",")

                results_list.append({
                    "Dateiname": filenames[i],
                    "Wahre Klasse": eval_dataset.id_to_label[true_idx],
                    "Vorhersage": eval_dataset.id_to_label[pred_idx],
                    "Status": "KORREKT" if pred_idx == true_idx else "FALSCH",
                    "Sicherheit_Prozent": round(confidences[i].item() * 100, 2),
                    "OCR_Text": clean_text
                })

    # Gesamte Accuracy
    acc = accuracy_score(all_labels, all_preds)
    print(f"\n========================================")
    print(f"MULTIMODAL ACCURACY (Test Set): {acc:.2%}")
    print(f"========================================\n")

    # Metriken berechnen
    class_names = [eval_dataset.id_to_label[i] for i in range(num_classes)]
    report_dict = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)

    # Detail-CSV speichern
    df_details = pd.DataFrame(results_list)
    save_path_csv = os.path.join(SAVE_DIR, OUTPUT_CSV)
    df_details.to_csv(save_path_csv, index=False, sep=';', encoding='utf-8-sig')

    # Metriken-CSV speichern
    metrics_list = []
    for name in class_names:
        metrics = report_dict[name]
        metrics_list.append({
            "Meme": name,
            "Precision": round(metrics['precision'], 2),
            "Recall": round(metrics['recall'], 2),
            "F1-Score": round(metrics['f1-score'], 2),
            "Anzahl": metrics['support']
        })

    df_metrics = pd.DataFrame(metrics_list)
    df_metrics = df_metrics.sort_values(by="F1-Score", ascending=True)
    df_metrics.to_csv(os.path.join(SAVE_DIR, "multimodal_metrics.csv"), index=False, sep=';')

    print("[FERTIG] Tabellen gespeichert. Deine Experimente sind komplett!")

if __name__ == "__main__":
    run_multimodal_evaluation()


--- Starte Multimodale Test-Evaluation ---
Lade offizielle Label-Map: multimodal_model_image_and_ocr_text_clip_map.json
--- [EVAL DATASET] Lade Test-Daten aus meme_test.json... ---


Lade Daten:   0%|          | 0/9637 [00:00<?, ?it/s]

-> Bereit: 9637 multimodale Test-Paare geladen.



pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

Lade Checkpoint: /content/drive/MyDrive/multimodal_image_and_ocr_NEW/multimodal_model_image_and_ocr_text_clip_epoch_20.pth
[SUCCESS] Gewichte erfolgreich geladen.
Berechne Vorhersagen auf ungesehenen Daten...


  0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in by


MULTIMODAL ACCURACY (Test Set): 55.86%

[FERTIG] Tabellen gespeichert. Deine Experimente sind komplett!


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
